In [1]:
#@title Setup (~5mins)

%pip install torch
%pip install pyrosetta-installer
!python -c 'import pyrosetta_installer; pyrosetta_installer.install_pyrosetta()'
%pip install biopython
%pip install tqdm
%pip install dm-tree
%pip install biotite
%pip install pyyaml
%pip install modelcif
%pip install torch_geometric
#!pip install boltz[cuda] -U
#%pip install torch_cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html
#!pip install easydict
#!pip install modelcif

!git clone https://github.com/nedru004/MPNN_affinity_maturation.git
#working_dir = "/Workspace/Users/david.nedrud@bio-techne.com/MPNN_affinity_maturation"
working_dir = "/content/MPNN_affinity_maturation"

!git clone https://github.com/dauparas/ProteinMPNN.git

!git clone https://gitlab.com/mjslee0921/flowpacker
#%pip install --use-deprecated=legacy-resolver -r flowpacker/requirements.txt

#!git clone https://github.com/cytokineking/ProtenixScore
#%cd $working_dir/MPNN_affinity_maturation/ProtenixScore
#!./install_protenixscore.sh --no-conda

!git clone https://github.com/ohuelab/boltzina.git


import sys
sys.path.insert(0, working_dir)
sys.path.insert(0, "/content/flowpacker")

Installing PyRosetta:
 os: ubuntu
 type: Release
 Rosetta C++ extras: 
 mirror: https://west.rosettacommons.org/pyrosetta/release/release
 extra packages: numpy

PyRosetta wheel url: https://:@west.rosettacommons.org/pyrosetta/release/release/PyRosetta4.Release.python312.ubuntu.wheel/pyrosetta-2026.6+release.e5a76a2dbd-cp312-cp312-linux_x86_64.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 854.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 111.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.2/399.2 kB 13.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 4.6 MB/s eta 0:00:00
  Created wheel for ihm: filename=ihm-2.9-

Cloning into 'MPNN_affinity_maturation'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 103 (delta 41), reused 73 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 17.51 MiB | 21.02 MiB/s, done.
Resolving deltas: 100% (41/41), done.
Cloning into 'ProteinMPNN'...
remote: Enumerating objects: 634, done.
remote: Counting objects: 100% (271/271), done.
remote: Compressing objects: 100% (125/125), done.
remote: Total 634 (delta 151), reused 146 (delta 146), pack-reused 363 (from 1)
Receiving objects: 100% (634/634), 119.90 MiB | 29.83 MiB/s, done.
Resolving deltas: 100% (290/290), done.
Cloning into 'flowpacker'...
remote: Enumerating objects: 214, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 214 (delta 34), reused 67 (delta 32), pack-reused 143 (from 1)
Receiving objects: 100% (214/214), 1.04 MiB 

In [2]:
#@title 1. MPNN Squence Design
import subprocess
import sys
import json # Added for JSON handling
import os   # Added for path operations

mpnn_script_path = f"ProteinMPNN/protein_mpnn_run.py" # Renamed 'script' to avoid conflict
folder_with_pdbs = f"{working_dir}/test/"
path_for_parsed_chains = f"{working_dir}/test/parsed_chains.jsonl"
path_for_assigned_chains = f"{working_dir}/test/assigned_chains.jsonl"
path_for_fixed_positions = f"{working_dir}/test/fixed_positions.jsonl" # Ensuring it is jsonl as per MPNN and code definition
path_for_fixed_positions_csv = f"{working_dir}/test/fixed_positions.csv"
out_dir = f"{working_dir}/test/mpnn_out/"
out_dir_bias = f"{working_dir}/test/mpnn_out_bias/"
bias_AA_file = f"{working_dir}/bias_AA.jsonl"
chains_to_design ="A"
num_seq_per_target=8
sampling_temp =0.5
seed = 37
fixed_positions="11 12 13 14 15" #fixing/not designing residues 1 2 3...25 in chain A and residues 10 11 12...40 in chain C
#generate fixed position csv
with open(path_for_fixed_positions_csv, 'w') as f:
  f.write(fixed_positions)

!python ProteinMPNN/helper_scripts/parse_multiple_chains.py --input_path=$folder_with_pdbs --output_path=$path_for_parsed_chains

!python ProteinMPNN/helper_scripts/assign_fixed_chains.py --input_path=$path_for_parsed_chains --output_path=$path_for_assigned_chains --chain_list "$chains_to_design"

!python ProteinMPNN/helper_scripts/make_fixed_positions_dict.py --input_path=$path_for_parsed_chains --output_path=$path_for_fixed_positions --chain_list "$chains_to_design" --position_list "$fixed_positions"

# Run actual protein_mpnn_run script
args_for_first_mpnn_run = [
    sys.executable, mpnn_script_path,
    "--jsonl_path", path_for_parsed_chains,
    "--chain_id_jsonl", path_for_assigned_chains,
    "--fixed_positions_jsonl", path_for_fixed_positions,
    "--out_folder", out_dir,
    "--num_seq_per_target", str(num_seq_per_target),
    "--sampling_temp", str(sampling_temp),
    "--seed", str(seed),
    "--batch_size", str(num_seq_per_target),
    "--omit_AAs", "C"]

subprocess.run(args_for_first_mpnn_run)


# MPNN with AA bias
# Ensure correct script path and flag for fixed positions.
args_for_second_mpnn_run = [
    sys.executable, mpnn_script_path,
    "--jsonl_path", path_for_parsed_chains,
    "--chain_id_jsonl", path_for_assigned_chains,
    "--fixed_positions_jsonl", path_for_fixed_positions,
    "--out_folder", out_dir_bias,
    "--num_seq_per_target", str(num_seq_per_target),
    "--sampling_temp", str(sampling_temp),
    "--seed", str(seed),
    "--batch_size", str(num_seq_per_target),
    "--omit_AAs", "C",
    "--bias_AA_jsonl", bias_AA_file]

subprocess.run(args_for_second_mpnn_run)

CompletedProcess(args=['/usr/bin/python3', 'ProteinMPNN/protein_mpnn_run.py', '--jsonl_path', '/content/MPNN_affinity_maturation/test/parsed_chains.jsonl', '--chain_id_jsonl', '/content/MPNN_affinity_maturation/test/assigned_chains.jsonl', '--fixed_positions_jsonl', '/content/MPNN_affinity_maturation/test/fixed_positions.jsonl', '--out_folder', '/content/MPNN_affinity_maturation/test/mpnn_out_bias/', '--num_seq_per_target', '8', '--sampling_temp', '0.5', '--seed', '37', '--batch_size', '8', '--omit_AAs', 'C', '--bias_AA_jsonl', '/content/MPNN_affinity_maturation/bias_AA.jsonl'], returncode=0)

In [3]:
#@title 2. Run flowpacker

from utilities import direct_fasta_to_csv

mpnn_folders = [f"{working_dir}/test/mpnn_out/seqs", f"{working_dir}/test/mpnn_out_bias/seqs"]
direct_fasta_to_csv(input_dirs=mpnn_folders, output_csv=f"{working_dir}/test/mpnn_final_result.csv", suffix=".pdb")

!python $working_dir/sampler_pdb_colab.py base --pdb_dir $working_dir/MPNN_affinity_maturation/test/mpnn_out --save_dir $working_dir/MPNN_affinity_maturation/test/flowpacker_out/ --csv_file $working_dir/MPNN_affinity_maturation/test/mpnn_final_result.csv

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
#@title 3. Score and filter binders for pTM and ipTM
# boltz score?
from utilities import filter_scores

# ProtenixScore
#!python $working_dir/protenix/score.py --input ./test/flowpacker_out/ --output ./test/protenix_score/ --recursive

# find all pdb
!boltz predict boltz_yaml_file --use_msa_server

# Filter scores
filter_scores("summary.csv", "test/filtered_pdb", threshold_pTM = 0.5, threshold_ipTM = 0.5)

In [ ]:
#@title 4. Rosetta interaction energy

#pdb_folder = "test"
pdb_file = f"{working_dir}/test/filtered_pdb"
interface_dist = 10
energy_filter = -5

!python $working_dir/per_residue_energy_pyrosetta.py --pdb $pdb_file --binder_id A --target_id M --interface_dist $interface_dist --output_dir $working_dir/test/ --xml_protocol $working_dir/update.xml

In [ ]:
#@title 5. Concatenate key residues

from MPNN_affinity_maturation.utilities import process_pdb_mutation_and_renumber
# merge key residues into one pdb file

# Extract Sequences
!python demo_scripts/pdb2fa.py $working_dir/test/af3score/filtered_links csv $working_dir/test/af3score/filtered_links.csv

process_pdb_mutation_and_renumber(f"{working_dir}/test/fixed_residue_pyrosetta.csv", f"{working_dir}/test/merge_motif_pdb/", renumber_chain='M', start_index=9) # The residue numbering of the target chain is consistent with the input target file